# Loading and Running Models from Saved Checkpoints

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from neural_models.dataset import NonogramsDataset
from neural_models.models.recursive_mlp import RecursiveMLP
from neural_models.models.recursive_grid_mlp import RecursiveGridMLP
from neural_models.criterions import AbstainLoss
from neural_models.train import Trainer
import itertools
from utils.display import display_grid, display_blended, display_blended_greyscale
from matplotlib import pyplot as plt

import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

In [ ]:
# Load model
# model_weights = torch.load("./checkpoints/nonogramsdataset/recursivemlp/21.pt")
# model_weights = torch.load("./checkpoints/nonogramsdataset/recursivegridmlp/default.pt")
model_weights = torch.load("./checkpoints/nonogramsdataset/recursivemlp/default.pt")
print(model_weights.keys())
# model = RecursiveGridMLP(input_shape=torch.Size((2, 5, 3)), output_shape=torch.Size((3, 5, 5)), hidden_size=256, num_layers=9, dropout=0.3)
model = RecursiveMLP(input_shape=torch.Size((2, 5, 3)), output_shape=torch.Size((3, 5, 5)), hidden_size=256, num_layers=9, dropout=0.3)
model.load_state_dict(model_weights)

In [ ]:
# Load dataset
train_dataset = NonogramsDataset("5x5-700_000-train.jsonl", max_size=10_000)
val_dataset = NonogramsDataset("5x5-100_000-val.jsonl", max_size=10_000)
test_dataset = NonogramsDataset("5x5-200_000-test.jsonl", max_size=10_000)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# Print train, validation, and test accuracy + loss
optimizer = torch.optim.Adam(model.parameters())
criterion = AbstainLoss()
trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, device="cpu")
train_loss, train_acc, train_additional = trainer.run_epoch(train_loader, train=False)
val_loss, val_acc, val_additional = trainer.run_epoch(val_loader, train=False)
test_loss, test_acc, test_additional = trainer.run_epoch(test_loader, train=False)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}, Train Additional: {train_additional}")
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}, Validation Additional: {val_additional}")
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}, Test Additional: {test_additional}")

### Incorrect Puzzles
Displaying Puzzles the model got incorrect

In [ ]:
model.eval()
n = 0
correct_clues, correct_grids, predicted_grids, output_grids = [], [], [], []
with torch.no_grad():
    for clues, mask, grids in itertools.chain.from_iterable(
        zip(*batch) for batch in test_loader
    ):
        outputs = model(clues.unsqueeze(0), mask=mask.unsqueeze(0))
        preds = criterion.predict(outputs).squeeze(0)          # (5,5) long
        if not (preds == grids).all():                          # same as trainer
            n += 1
            correct_clues.append(clues)
            correct_grids.append(grids)
            predicted_grids.append(preds)
            output_grids.append(outputs.squeeze(0))

        if n == 12:
            break
print(correct_clues[0])
print(correct_grids[0])
print(predicted_grids[0])
print(output_grids[0])

In [ ]:
figs, axs = plt.subplots(3, 4, figsize=(12, 9))
for i in range(3):
    for j in range(4):
        idx = i * 4 + j
        if idx >= len(correct_clues):
            break
        display_grid(ax=axs[i, j], grid=correct_grids[idx], clues=correct_clues[idx])
figs, axs = plt.subplots(3, 4, figsize=(12, 9))
for i in range(3):
    for j in range(4):
        idx = i * 4 + j
        if idx >= len(correct_clues):
            break
        display_grid(ax=axs[i, j], grid=predicted_grids[idx], clues=correct_clues[idx])
figs, axs = plt.subplots(3, 4, figsize=(12, 9))
for i in range(3):
    for j in range(4):
        idx = i * 4 + j
        if idx >= len(correct_clues):
            break
        display_blended_greyscale(ax=axs[i, j], grid=output_grids[idx], clues=correct_clues[idx])
figs, axs = plt.subplots(3, 4, figsize=(12, 9))
for i in range(3):
    for j in range(4):
        idx = i * 4 + j
        if idx >= len(correct_clues):
            break
        display_blended(ax=axs[i, j], grid=output_grids[idx], clues=correct_clues[idx])


### Logit Lenses

In [ ]:
clue, mask, grid = test_dataset[0]
ax = plt.subplot(1, 3, 1)
display_grid(ax=ax, grid=grid, clues=clue)

model.eval()
intermediate_predictions = []
n = 16
for i in range(n):
    intermediate_predictions.append(model(clue.unsqueeze(0), mask=mask.unsqueeze(0), layer_num=i).squeeze(0))
figs, axs = plt.subplots(n // 4, 4, figsize=(12, 3 * (n // 4)))
for i in range(n // 4):
    for j in range(4):
        idx = i * 4 + j
        if idx >= len(intermediate_predictions):
            break
        prediction = criterion.predict(intermediate_predictions[idx].unsqueeze(0)).squeeze(0)
        display_grid(ax=axs[i, j], grid=prediction, clues=clue)
        display_blended(ax=axs[i, j], grid=intermediate_predictions[idx], clues=clue)